In [ ]:
# --- Governed engine install (mirrors the training notebook's frozen set) ---
# The training kernel installed the governed freeze; this inference kernel must do
# the same, or `unsloth` is absent (version 1 failed with ModuleNotFoundError).
import subprocess, sys, os, shutil
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
UV = shutil.which('uv') or 'uv'
BASE = ['uv', 'pip', 'install', '--system', '--python', sys.executable, '--no-cache-dir']
RESOLVER = ['torch>=2.8.0', 'triton>=3.4.0', 'transformers==4.56.2', 'peft==0.20.0',
            'trl==0.22.2', 'datasets==5.0.1', 'accelerate==1.15.0',
            'bitsandbytes==0.50.2', 'openai-harmony==0.0.8']
FROZEN_NO_DEPS = ['unsloth==2026.9.4', 'unsloth_zoo==2026.9.3']
def run(cmd, phase):
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(phase, '-> exit', p.returncode)
    if p.returncode != 0:
        print(p.stdout[-1500:]); print(p.stderr[-1500:])
        raise RuntimeError(phase + ' failed')
run([UV, 'pip', 'install', '--system', '--python', sys.executable, '--no-cache-dir', *RESOLVER], 'resolver')
run([UV, 'pip', 'install', '--system', '--python', sys.executable, '--no-cache-dir', '--no-deps',
     '--upgrade', *FROZEN_NO_DEPS], 'frozen-no-deps')
print('engine install complete')


In [ ]:
# GHARIBO EXP-002 QUALIFICATION - INFERENCE ONLY
#
# INFERENCE ONLY. No gold answers, no scoring data, no TEST payload. Only
# predictions travel back; scoring happens locally.
import json, torch, pathlib

PROMPTS = json.loads(r'''[{"item_id": "69797dbc393bc043080c4f3c10f46e213c45b190547197b0756ea44f78547d5e", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:site-remediation) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:site-remediation\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Site Remediation\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Site Remediation is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on site remediation; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Construction\",\n        \"Environmental & Waste Management\",\n        \"Government & Public Sector\",\n        \"Industrial & Manufacturing\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Contaminated-site cleanup\",\n        \"Groundwater remediation\",\n        \"Soil remediation\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"C\",\n      \"standardsOrClassificationEvidence\": [\n        \"UN International Standard Industrial Classification (ISIC Rev. 5)\",\n        \"2022 North American Industry Classification System\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:425248a022b6302b5b3b3e94\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the technical/commercial scope represented by Core Domain Site Remediation.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:545a6363a1ffac46a2689f8e\",\n        \"sourceUrl\": \"https://www.census.gov/naics/\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"2022 North American Industry Classification System supports the technical/commercial scope represented by Core Domain Site Remediation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "89e40c4753e380d4e871c84bf44731498648c0289a2aac659e16413a358b3080", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:belongs_to_category:29726853a2daf172) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:belongs_to_category:29726853a2daf172\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BELONGS_TO_CATEGORY\",\n      \"sourceExternalKey\": \"model:dahua:dahua-wizmind-5-series-smart-dual-light-2mp:ipc-hdw5259t-ze-il\",\n      \"targetExternalKey\": \"category:security:dome-camera\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:dahua-wm5-sdl2\",\n        \"sourceUrl\": \"https://www.dahuasecurity.com/mena/Products/All-Products/Network-Cameras/WizMind-5-Series/Smart-Dual-Light\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BELONGS_TO_CATEGORY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "1ddfcfb3b982ac980879445b96596978fadaa3f7f0e1c684edc41a2df68d23a2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:global-map:child-of:25e43ffef9cc8d58fae3) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:global-map:child-of:25e43ffef9cc8d58fae3\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"relationType\": \"CHILD_OF\",\n      \"sourceExternalKey\": \"domain:global:hydropower-systems\",\n      \"targetExternalKey\": \"category:global:energy-and-power\",\n      \"sourceEntityType\": \"DOMAIN\",\n      \"targetEntityType\": \"CATEGORY\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"Hydropower Systems is commercially valid within Energy & Power; the Core Domain may belong to multiple Categories.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:45365487d140c12dfe8be563\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports a commercially valid relationship between Core Domain Hydropower Systems and Category Energy & Power.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "1e8aba1a8635e1085c22a102e97731a22e738cec0e94acabf6aa0f3f1dbb8e46", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:hid:hid-signo-biometric) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:hid:hid-signo-biometric\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"HID\",\n      \"brand\": \"HID\",\n      \"family\": \"HID Signo Biometric\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Biometric Access Reader\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"HID Signo Biometric\",\n      \"description\": \"Official HID product/compliance source family grouping for physical access-control hardware.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:hid-25b\",\n        \"sourceUrl\": \"https://www.hidglobal.com/products/25b\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the HID Signo Biometric product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b7dc8ca567e861ae7ab788d6c7fa63b63f77e0843f964aace7aa52acc34e09c8", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:gas-detection-systems) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:gas-detection-systems\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Gas Detection Systems\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Gas Detection Systems is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on gas detection systems; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Chemicals & Process Industries\",\n        \"Industrial & Manufacturing\",\n        \"Oil, Gas & Petrochemicals\",\n        \"Security & Public Safety\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Fixed gas detection\",\n        \"Gas controllers\",\n        \"Portable gas detection\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"I\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:58e0465310b524bf1b9b5eee\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Gas Detection Systems.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:bad014e1afac7dc5a3817265\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Gas Detection Systems.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "2a3409be04b506537f21a46942a6d574425ff95caa74c0044fd18fb4b828537d", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:home-automation) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:home-automation\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Home Automation\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Home Automation is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on home automation; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Construction\",\n        \"Consumer & Household\",\n        \"ICT & Technology\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Connected lighting\",\n        \"Home controls\",\n        \"Smart home hubs\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"G\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"ETIM Classification Model\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:686aff604dfb047b0cff9fe9\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Home Automation.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:143a4eed436ad3ade043e7ce\",\n        \"sourceUrl\": \"https://www.etim-international.com/classification/model-information/\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ETIM Classification Model supports the technical/commercial scope represented by Core Domain Home Automation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "c1a67234d4ac4c4442f831d803b810c80954902f6f32c47b307d912dd63500bd", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:hikvision:ds-76-recorder-family:ds-7632ni-i2) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:hikvision:ds-76-recorder-family:ds-7632ni-i2\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Hikvision\",\n      \"brand\": \"Hikvision\",\n      \"family\": \"DS-76 Recorder Family\",\n      \"series\": \"DS-76 Recorder Family\",\n      \"model\": \"DS-7632NI-I2\",\n      \"modelNumber\": \"DS-7632NI-I2\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"DS-7632NI-I2\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"subcategory\": null,\n      \"productRole\": \"Video recorder / NVR\",\n      \"systemRole\": \"NVR-based IP Surveillance System\",\n      \"lifecycle\": \"UNKNOWN\",\n      \"name\": \"DS-7632NI-I2\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"DS-7632NI-I2\",\n        \"issuer\": \"Hikvision\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:53c59e37a28bb0459b4fab60\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=5\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity DS-7632NI-I2.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a044668699e3a7d9a4c1c54df552b7de7c3d04b7147c46621603c6991f340e21", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:ubiquiti:unifi-protect-bullet-cameras:g5-pro) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:ubiquiti:unifi-protect-bullet-cameras:g5-pro\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Ubiquiti\",\n      \"brand\": \"UniFi\",\n      \"family\": \"UniFi Protect Bullet Cameras\",\n      \"model\": \"G5 Pro\",\n      \"modelNumber\": \"G5 Pro\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"G5 Pro\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Bullet Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"IP video surveillance camera\",\n      \"systemRole\": \"Video capture / UniFi Protect endpoint\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"G5 Pro\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"G5 Pro\",\n        \"issuer\": \"Ubiquiti\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:ubi-physical\",\n        \"sourceUrl\": \"https://www.ui.com/physical-security\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports model identity G5 Pro.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b03db5dde8d8c264aa1137b96cc1a6276d63230b6e20e644845e265c2d2b01d2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:dahua:dahua-discontinued-network-cameras:hero-h5as) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:dahua:dahua-discontinued-network-cameras:hero-h5as\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Dahua Technology\",\n      \"brand\": \"Dahua\",\n      \"family\": \"Dahua Discontinued Network Cameras\",\n      \"series\": \"Dahua Discontinued Network Cameras\",\n      \"model\": \"Hero H5AS\",\n      \"modelNumber\": \"Hero H5AS\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"Hero H5AS\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"Historical network surveillance camera\",\n      \"systemRole\": \"IP Video Surveillance System\",\n      \"lifecycle\": \"LEGACY\",\n      \"name\": \"Hero H5AS\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"Hero H5AS\",\n        \"issuer\": \"Dahua Technology\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:76bed837b3c2cc0221923c11\",\n        \"sourceUrl\": \"https://www.dahuasecurity.com/sa/Products/All-Products/Discontinued-Products/Network-Cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity Hero H5AS.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "c02027ccd0d5d578f5fb8550eade916563694bde9234a2c4aec4ea01de9ad7da", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:electric-motors-and-drives) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:electric-motors-and-drives\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Electric Motors & Drives\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Electric Motors & Drives is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on electric motors & drives; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Energy & Power\",\n        \"Industrial & Manufacturing\",\n        \"Oil, Gas & Petrochemicals\",\n        \"Water & Wastewater\"\n      ],\n      \"likelySystemFamilies\": [\n        \"LV motors\",\n        \"MV motors\",\n        \"Variable-frequency drives\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:f42b54ebe17530c6ce9661d1\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Electric Motors & Drives.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:d283c9710af0d99ac154188e\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Electric Motors & Drives.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b296ff4e159e2281bc50782963cc021f3be605424498b2dee9099cfd1de602e0", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:humanitarian-wash-and-sanitation) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:humanitarian-wash-and-sanitation\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:35:41+03:00\",\n    \"payload\": {\n      \"name\": \"Humanitarian WASH & Sanitation\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Humanitarian WASH & Sanitation is a reusable commercial/technical specialization in the VOKA global library, grouping real systems, products and services around this functional scope.\",\n      \"boundary\": \"Centered on humanitarian wash & sanitation; adjacent categories and domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Humanitarian & Development Operations\",\n        \"Water & Wastewater\",\n        \"Environmental & Waste Management\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Emergency water supply\",\n        \"Sanitation\",\n        \"Hygiene support\",\n        \"Field WASH\"\n      ],\n      \"relevantGlobalServiceFamilies\": [],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"K\",\n      \"standardsOrClassificationEvidence\": [\n        \"UNHCR — Become a Supplier / Goods and Services Usually Procured\",\n        \"UNGM — Humanitarian Relief Items, Kits or Accessories\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:99024cdfb527b47245b48bc0\",\n        \"sourceUrl\": \"https://www.unhcr.org/get-involved/work-us/become-supplier\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:35:41+03:00\",\n        \"claim\": \"UNHCR — Become a Supplier / Goods and Services Usually Procured supports the technical/commercial scope represented by Core Domain Humanitarian WASH & Sanitation.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:076b6eabe2f5af09c632d51c\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/Notice/301717\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:35:41+03:00\",\n        \"claim\": \"UNGM — Humanitarian Relief Items, Kits or Accessories supports the technical/commercial scope represented by Core Domain Humanitarian WASH & Sanitation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "945f273cd5979eec7b656d94dddf122143fccbce49b0914b6b0cfd045bbcfb4b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:power-transmission) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:power-transmission\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Power Transmission\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Power Transmission is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on power transmission; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Civil Infrastructure\",\n        \"Energy & Power\",\n        \"Utilities & Municipal Services\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Grid protection\",\n        \"HV transmission\",\n        \"Transmission substations\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"C\",\n      \"standardsOrClassificationEvidence\": [\n        \"Central Product Classification Version 2.1\",\n        \"UN International Standard Industrial Classification (ISIC Rev. 5)\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:acaa8918471269114ac26fdb\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/CPC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"Central Product Classification Version 2.1 supports the technical/commercial scope represented by Core Domain Power Transmission.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:3e914b763c1688f538a53a4b\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the technical/commercial scope represented by Core Domain Power Transmission.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "09d6e1a3b9a383959aa2739cde1e71cff21d2b317a6185e22bc8851db54136f6", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:component_of:0e8c39fa677fe6015066) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:component_of:0e8c39fa677fe6015066\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"COMPONENT_OF\",\n      \"sourceExternalKey\": \"model:i-pro:i-pro-s-series:wv-s65340-z2\",\n      \"targetExternalKey\": \"system:security:ip-video-surveillance\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"WV-S65340-Z2 is commercially relevant as a component of IP Video Surveillance System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:1413a89618589216737eb920\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=S-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"WV-S65340-Z2 is commercially relevant as a component of IP Video Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "2ceab483da296cce1d6513f37f7a34f02e69b9d0020fada13265a253bca6698a", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:axis:axis-q35-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:axis:axis-q35-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Axis Communications\",\n      \"brand\": \"AXIS\",\n      \"family\": \"AXIS Q35 Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Dome Camera\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"AXIS Q35 Series\",\n      \"description\": \"High-performance robust fixed dome camera series.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"impact_rating\",\n        \"name\": \"Impact resistance\",\n        \"value\": \"IK10\",\n        \"unit\": null,\n        \"group\": \"Environmental\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"ingress_ratings\",\n        \"name\": \"Ingress ratings\",\n        \"value\": \"IP66; IP6K9K; NEMA 4X\",\n        \"unit\": null,\n        \"group\": \"Environmental\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"operating_temp_min\",\n        \"name\": \"Operating temperature minimum\",\n        \"value\": -55,\n        \"unit\": \"°C\",\n        \"group\": \"Environmental\",\n        \"dataType\": \"number\"\n      },\n      {\n        \"code\": \"operating_temp_max\",\n        \"name\": \"Operating temperature maximum\",\n        \"value\": 55,\n        \"unit\": \"°C\",\n        \"group\": \"Environmental\",\n        \"dataType\": \"number\"\n      },\n      {\n        \"code\": \"power_redundancy\",\n        \"name\": \"Power redundancy\",\n        \"value\": \"DC and PoE\",\n        \"unit\": null,\n        \"group\": \"Power\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"key_protection\",\n        \"name\": \"Secure key storage\",\n        \"value\": \"FIPS 140-3 Level 3\",\n        \"unit\": null,\n        \"group\": \"Cybersecurity\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:axis-q35\",\n        \"sourceUrl\": \"https://www.axis.com/products/axis-q35-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the AXIS Q35 Series product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "479c1c9f04c5ca0fb3fc30854d26d4dd5cc0ae1eae5f32f0d911a7a5fcd96445", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:hikvision:hikvision-ds-2cd2543g2-mini-dome) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:hikvision:hikvision-ds-2cd2543g2-mini-dome\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Hikvision\",\n      \"brand\": \"Hikvision\",\n      \"family\": \"Hikvision DS-2CD2543G2 Mini Dome\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Dome Camera\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"Hikvision DS-2CD2543G2 Mini Dome\",\n      \"description\": \"Official Hikvision product/support grouping used for this batch; family grouping follows documented model prefix/generation.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:hik-2543li\",\n        \"sourceUrl\": \"https://pro-av.hikvision.com/mena-en/products/IP-Products/Network-Cameras/Pro-Series-EasyIP-/ds-2cd2543g2-li-w--s--2u-/\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Hikvision DS-2CD2543G2 Mini Dome product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "1e07dec1592b9f650acc20cc0c8a3b01b42a4d14a7973775b266a1cdee0c2765", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:parent_of:402b3b11d1d75f703cf3) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:parent_of:402b3b11d1d75f703cf3\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"family:i-pro:i-pro-x-series\",\n      \"targetExternalKey\": \"model:i-pro:i-pro-x-series:wv-x66600-z3k\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"i-PRO X Series contains the verified model WV-X66600-Z3K.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:20e8e19204c445176bb2a267\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=X-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"i-PRO X Series contains the verified model WV-X66600-Z3K.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "38b107eea849d39e07d20d4c8d1da17b9eebec18b40691b21e09269f6fb98fcf", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:industrial-networking) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:industrial-networking\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Industrial Networking\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Industrial Networking is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on industrial networking; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Energy & Power\",\n        \"ICT & Technology\",\n        \"Industrial & Manufacturing\",\n        \"Oil, Gas & Petrochemicals\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Field networks\",\n        \"Industrial Ethernet\",\n        \"Industrial wireless\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"ETIM Classification Model\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:2c9c085cac5240ff7b0ac8fd\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Industrial Networking.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:b1a2622556e8372c98d05087\",\n        \"sourceUrl\": \"https://www.etim-international.com/classification/model-information/\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ETIM Classification Model supports the technical/commercial scope represented by Core Domain Industrial Networking.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "c118620fe53cfbca6db6380cdde8615163049a287d824854f328a29c019df73d", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:payment-and-transaction-infrastructure) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:payment-and-transaction-infrastructure\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Payment & Transaction Infrastructure\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Payment & Transaction Infrastructure is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on payment & transaction infrastructure; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Financial Services & Insurance\",\n        \"ICT & Technology\",\n        \"Retail & Commerce\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Payment processing\",\n        \"Payment terminals\",\n        \"Transaction systems\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"G\",\n      \"standardsOrClassificationEvidence\": [\n        \"Central Product Classification Version 2.1\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:5f4d838e387206fb9e967e2e\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/CPC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"Central Product Classification Version 2.1 supports the technical/commercial scope represented by Core Domain Payment & Transaction Infrastructure.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:d87527a860818cf6bba22dea\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Payment & Transaction Infrastructure.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "e11e5f305db5500db9e015395732d84967e778205e3104208d2e5ffac564cf95", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:transport-terminals) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:transport-terminals\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Transport Terminals\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Transport Terminals is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on transport terminals; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Civil Infrastructure\",\n        \"Logistics & Warehousing\",\n        \"Transportation & Mobility\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Freight terminals\",\n        \"Intermodal terminals\",\n        \"Passenger terminals\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"A\",\n      \"standardsOrClassificationEvidence\": [\n        \"Central Product Classification Version 2.1\",\n        \"UN International Standard Industrial Classification (ISIC Rev. 5)\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:4aa07465401b412ae4f771c4\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/CPC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"Central Product Classification Version 2.1 supports the technical/commercial scope represented by Core Domain Transport Terminals.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:400ec78c096cfc28621ea355\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the technical/commercial scope represented by Core Domain Transport Terminals.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "df7534b65cda8217bb4f632ff49ebbbe33dae0d4065316f1b8c60e812d57a789", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:dahua:dahua-wizsense-nvr-5-ei-8hdd) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:dahua:dahua-wizsense-nvr-5-ei-8hdd\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Dahua Technology\",\n      \"brand\": \"Dahua\",\n      \"family\": \"Dahua WizSense NVR 5-EI 8HDD\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"lifecycle\": null,\n      \"name\": \"Dahua WizSense NVR 5-EI 8HDD\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:9448a8acc52c4030dcb6\",\n        \"sourceUrl\": \"https://www.dahuasecurity.com/mena/Products/All-Products/Network-Recorders/WizSense-Series/NVR-5-EI-Series/8HDD\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping Dahua WizSense NVR 5-EI 8HDD.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b52e0790f4704d55858016b9fd6f8cc3ec97042fc890fd7b8cffffedc22c6c2e", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:88d1e52ce6710a5f) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:88d1e52ce6710a5f\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:hanwha:hanwha-qnd-60x2r-support-group:qnd-6012r\",\n      \"targetExternalKey\": \"brand:hanwha-vision\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:hanwha-qnd6022r\",\n        \"sourceUrl\": \"https://supportportal.hanwhavision.com/global/products/QND-6022R-en\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BRANDED_BY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "2200c67d32bb98465bc48ff12a98ecc9444fd6d946693a489814f53bb2211bb8", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:grain-handling-and-storage) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:grain-handling-and-storage\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Grain Handling & Storage\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Grain Handling & Storage is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on grain handling & storage; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Agriculture & Agribusiness\",\n        \"Food & Beverage Processing\",\n        \"Logistics & Warehousing\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Drying\",\n        \"Grain conveyors\",\n        \"Silos\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"G\",\n      \"standardsOrClassificationEvidence\": [\n        \"UNGM Transportation and Storage Services\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:b23a391a770bf4c87ae62123\",\n        \"sourceUrl\": \"https://www.ungm.org/Shared/KnowledgeCenter/Pages/PC_TranStorMail\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNGM Transportation and Storage Services supports the technical/commercial scope represented by Core Domain Grain Handling & Storage.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:ccdb19f0b95674886f5c91d7\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Grain Handling & Storage.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "0ba8dae3ab8f6680657604bc616da3bc73ea7fe0a4f39be18e64408af14821c7", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:multifactor-access-control) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:multifactor-access-control\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Multi-factor Access Control System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"AUTHENTICATION_ARCHITECTURE\",\n      \"purpose\": \"Access architecture requiring multiple supported credential/authentication factors, such as card plus PIN or biometric combinations.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Multi-factor-capable terminal/readers\",\n        \"Access controller/software\",\n        \"Credential/identity store\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Explicit product/system support for the selected factor combination\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"AUTHENTICATION_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:access-control\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:736e678c654b61e9a63d0784\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Multi-factor Access Control System.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:security-registry:74706c9840f3e0e711cbc5ee\",\n        \"sourceUrl\": \"https://support.supremainc.com/en/support/solutions/articles/24000106273-suprema-new-and-updated-article-announcement-releases-july-2026\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Multi-factor Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "e44650f2658617f7cd5d8cf158dcb79526761102b2c8455e07e9f110f3b6ff5b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:security-registry:parent_of:7733b07a0fc011272e6b) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:security-registry:parent_of:7733b07a0fc011272e6b\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"system:security:ip-video-surveillance\",\n      \"targetExternalKey\": \"system:security:onvif-interoperable-ip-video\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"IP Video Surveillance System is the parent architecture for ONVIF-interoperable IP Video System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:79067ab5df616882df208380\",\n        \"sourceUrl\": \"https://www.onvif.org/profiles/\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"IP Video Surveillance System is the parent architecture for ONVIF-interoperable IP Video System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "9c5335a5bef0ec83cb4a38c6e3b07981a7f0bb5fc647c6bda07fff09dfa2e4f4", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:manufactured_by:a1e795d2599fcce6) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:manufactured_by:a1e795d2599fcce6\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"MANUFACTURED_BY\",\n      \"sourceExternalKey\": \"model:axis:axis-m20-series:axis-m2048-le\",\n      \"targetExternalKey\": \"manufacturer:axis\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:axis-m20\",\n        \"sourceUrl\": \"https://www.axis.com/products/axis-m20-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports MANUFACTURED_BY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "148a15a72b3e5019d2004ae8d2161826004c9777cce5e3b191fd2291968f0ed9", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:parent_of:127cd3aece6f926f4628) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:parent_of:127cd3aece6f926f4628\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"family:dahua:dahua-project-exclusive-network-cameras\",\n      \"targetExternalKey\": \"model:dahua:dahua-project-exclusive-network-cameras:ipc-hdbw5859z-zhe-pv-pro-atc-v1\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"Dahua Project Exclusive Network Cameras contains the verified model IPC-HDBW5859Z-ZHE-PV-PRO-ATC(V1).\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:0dad3ea5b811829b322fcc59\",\n        \"sourceUrl\": \"https://previous.dahuasecurity.com/Products/All-Products/Dedicated-Products/Project-Exclusive/Network-Cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Dahua Project Exclusive Network Cameras contains the verified model IPC-HDBW5859Z-ZHE-PV-PRO-ATC(V1).\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b303b8ad2c9a3b8abda0db5ef39743e38b603517e618bb03c65a0833dffb5e47", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:industrial-boilers-and-steam-systems) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:industrial-boilers-and-steam-systems\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Industrial Boilers & Steam Systems\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Industrial Boilers & Steam Systems is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on industrial boilers & steam systems; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Chemicals & Process Industries\",\n        \"Energy & Power\",\n        \"Food & Beverage Processing\",\n        \"Healthcare\",\n        \"Industrial & Manufacturing\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Industrial boilers\",\n        \"Steam distribution\",\n        \"Steam generation\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"standardsOrClassificationEvidence\": [\n        \"Central Product Classification Version 2.1\",\n        \"ECLASS Basic 16.0 Public Content Search\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:a60cf5a488145d9313fe1d39\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/CPC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"Central Product Classification Version 2.1 supports the technical/commercial scope represented by Core Domain Industrial Boilers & Steam Systems.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:4ad98a1f1f2b51a598689a31\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Industrial Boilers & Steam Systems.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "4994c9da66ae344decb951076ba3c90702a5ef90b9a380062f6e51cd1e10ddee", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:wireless-lock-access-control) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:wireless-lock-access-control\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Wireless Electronic-lock Access Control System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"WIRELESS_LOCK_ARCHITECTURE\",\n      \"purpose\": \"Networked access architecture integrating supported wireless electronic locks through gateways/controllers.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Wireless electronic locks\",\n        \"Wireless/IP gateway or supported controller\",\n        \"Access software\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Explicit lock/gateway/controller/software compatibility\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"WIRELESS_LOCK_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:access-control\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:87c61e3c4feaa49a0f4c2047\",\n        \"sourceUrl\": \"https://synergis-cloudlink-help.genetec.com/EN/EN/SSW/T_SSW_Enrolling_AllegionSchlageIPLocks.html\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Wireless Electronic-lock Access Control System.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:security-registry:22d1ab4a2dacc481aa2ed9a4\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Wireless Electronic-lock Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "8ad1131619749e328ef01c06d51e9201ab5cf8aa853ceef1fee192a5f3006e85", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:security:third-party-system-integration) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:security:third-party-system-integration\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Third-party Security System Integration\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SERVICE\",\n      \"serviceClass\": \"SYSTEM_INTEGRATION\",\n      \"purpose\": \"Integrate supported third-party security devices/applications with a security platform.\",\n      \"typicalActivities\": [\n        \"Review integration interface\",\n        \"Configure supported integration\",\n        \"Validate end-to-end behavior\"\n      ],\n      \"typicalDeliverables\": [\n        \"Operational integration\"\n      ],\n      \"applicableSystemGroups\": [\n        \"ALL_SECURITY_SYSTEMS\"\n      ],\n      \"deliveryModes\": [\n        \"Remote\",\n        \"On-site\"\n      ],\n      \"recurring\": false,\n      \"prerequisites\": [],\n      \"vendorNeutralRegistryService\": true,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"service_class\",\n        \"name\": \"Service class\",\n        \"value\": \"SYSTEM_INTEGRATION\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:services:integration\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recurring\",\n        \"name\": \"Recurring service\",\n        \"value\": false,\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"boolean\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:390eedc2586c14ec200fd718\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Third-party Security System Integration.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:security-registry:16662712404226b94e567e7c\",\n        \"sourceUrl\": \"https://www.milestonesys.com/products/software/xprotect/\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Third-party Security System Integration.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:security-registry:4330937b968e19f8d7b4c956\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/solutions/professional-services\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Third-party Security System Integration.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "084e1d08cedb1731cf5e36a3ba0e4800e287ae39a7ba4ec4bb23c29c41da637f", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:dahua:dahua-distribution-wizsense-network-cameras:ipc-ufw3459t-zas-il) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:dahua:dahua-distribution-wizsense-network-cameras:ipc-ufw3459t-zas-il\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Dahua Technology\",\n      \"brand\": \"Dahua\",\n      \"family\": \"Dahua Distribution WizSense Network Cameras\",\n      \"series\": \"Dahua Distribution WizSense Network Cameras\",\n      \"model\": \"IPC-UFW3459T-ZAS-IL\",\n      \"modelNumber\": \"IPC-UFW3459T-ZAS-IL\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"IPC-UFW3459T-ZAS-IL\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"Network surveillance camera\",\n      \"systemRole\": \"IP Video Surveillance System\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"IPC-UFW3459T-ZAS-IL\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"IPC-UFW3459T-ZAS-IL\",\n        \"issuer\": \"Dahua Technology\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:8ef334791729b44b169a4bcd\",\n        \"sourceUrl\": \"https://www.dahuasecurity.com/ceen/Products/All-Products/Dedicated-Products/Distribution-Products/Network-Cameras/Wizsense-Series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity IPC-UFW3459T-ZAS-IL.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "73a65e798efda5848ed75a9c9364f68550b2d41c2e5af65e250ccf608efa8680", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:component_of:812ff5faad9c49f9470f) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:component_of:812ff5faad9c49f9470f\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"COMPONENT_OF\",\n      \"sourceExternalKey\": \"model:hikvision:ds-81-recorder-family:ds-8116hqhi-f8-n\",\n      \"targetExternalKey\": \"system:security:nvr-ip-surveillance\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"DS-8116HQHI-F8/N is commercially relevant as a component of NVR-based IP Surveillance System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:6cb05b3b487de2a92e5ccdb5\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=3\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"DS-8116HQHI-F8/N is commercially relevant as a component of NVR-based IP Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "9fa420fdef03bb0d60c1308f9a9b8602e327eadc094076c947a454664e7695d7", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:ubiquiti:unifi-protect) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:ubiquiti:unifi-protect\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Ubiquiti\",\n      \"brand\": \"UniFi\",\n      \"name\": \"UniFi Protect\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"serviceRole\": \"Video management and recording platform\",\n      \"lifecycle\": \"CURRENT\",\n      \"description\": \"UniFi physical-security platform for local video management/recording; official physical-security page states no licensing fees and support for third-party ONVIF cameras.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"licensing\",\n        \"name\": \"Licensing fee\",\n        \"value\": \"No licensing fees\",\n        \"unit\": null,\n        \"group\": \"Commercial\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"third_party_camera\",\n        \"name\": \"Third-party camera support\",\n        \"value\": \"ONVIF\",\n        \"unit\": null,\n        \"group\": \"Interoperability\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:ubi-physical\",\n        \"sourceUrl\": \"https://www.ui.com/physical-security\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official Ubiquiti physical-security page describes UniFi Protect platform, local recording and ONVIF support.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "897596647c025ccf011965c35366c2662d8e4a41a672985a1852693600a42346", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:axis:axis-q62-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:axis:axis-q62-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Axis Communications\",\n      \"brand\": \"AXIS\",\n      \"family\": \"AXIS Q62 Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"PTZ Camera\",\n      \"lifecycle\": null,\n      \"name\": \"AXIS Q62 Series\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:2ffb34739356d6a913ff\",\n        \"sourceUrl\": \"https://www.axis.com/products/ptz-cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping AXIS Q62 Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "dfa4a8ecba5695a50cee3900979769434dd91dd66105edeaf7a68425a51724cc", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:70e83a5bd7e7c767e5be) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:70e83a5bd7e7c767e5be\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:hikvision:ds-96-recorder-family:ds-9632n-m8\",\n      \"targetExternalKey\": \"brand:hikvision\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"DS-9632N-M8 is marketed under the Hikvision brand.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:55516d11dd7785af84abc6fa\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=1\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"DS-9632N-M8 is marketed under the Hikvision brand.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "aa0a172e12b5c35e3e9a977286d65f42ce9ad21f23cb113b48ed97f5d1e7863b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:utility-corridors-and-duct-banks) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:utility-corridors-and-duct-banks\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Utility Corridors & Duct Banks\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Utility Corridors & Duct Banks is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on utility corridors & duct banks; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Civil Infrastructure\",\n        \"Energy & Power\",\n        \"Telecommunications\",\n        \"Utilities & Municipal Services\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Cable corridors\",\n        \"Shared utility infrastructure\",\n        \"Utility ducts\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"A\",\n      \"standardsOrClassificationEvidence\": [\n        \"Central Product Classification Version 2.1\",\n        \"UN International Standard Industrial Classification (ISIC Rev. 5)\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:6a8a0a2d9561cbd9bb0fc17c\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/CPC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"Central Product Classification Version 2.1 supports the technical/commercial scope represented by Core Domain Utility Corridors & Duct Banks.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:4106b11b7c34da5f152fb75f\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the technical/commercial scope represented by Core Domain Utility Corridors & Duct Banks.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "d0c064337d2035d4c048dd1d7ea0e176247e0698dea3830fa56fe7b4dc6cd3bc", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:b1a70f49b9b5e2014a54) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:b1a70f49b9b5e2014a54\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:tp-link:omada-agile-switches:es208g\",\n      \"targetExternalKey\": \"brand:omada\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"ES208G is marketed under the Omada brand.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:3d397736b690cba4b260632b\",\n        \"sourceUrl\": \"https://www.omadanetworks.com/us/business-networking/all-omada-switch/\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"ES208G is marketed under the Omada brand.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "475b33ab5b106a15c7b976031afd203698e34e5eddee773b6a2c6efe1e627692", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:hid:hid-signo-readers) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:hid:hid-signo-readers\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"HID\",\n      \"brand\": \"HID\",\n      \"family\": \"HID Signo Readers\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"RFID / Mobile Credential Reader\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"HID Signo Readers\",\n      \"description\": \"Official HID product/compliance source family grouping for physical access-control hardware.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:hid-signo\",\n        \"sourceUrl\": \"https://www.hidglobal.com/product-mix/signo-readers\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the HID Signo Readers product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b2315f0178700c7abb145fb5873f42b3701c1eb1a46be253cb36793b1c4b0934", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:parent_of:849a9048491976a1e394) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:parent_of:849a9048491976a1e394\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"family:hikvision:ds-77-recorder-family\",\n      \"targetExternalKey\": \"model:hikvision:ds-77-recorder-family:ds-7732ni-e4\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"DS-77 Recorder Family contains the verified model DS-7732NI-E4.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:573be7b208f6f33e25171a8d\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=6\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"DS-77 Recorder Family contains the verified model DS-7732NI-E4.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "f7361a16818c9f26473dad7db75e465a1c90f4b942778347440c7184fbb81cf4", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:938dd5058f20dafc) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:938dd5058f20dafc\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"family:suprema:xpass-d2\",\n      \"targetExternalKey\": \"brand:suprema\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector2\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=2&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BRANDED_BY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "9c334b32a504a8d3da8a3def20da597c2800261e314a7cbf56ac72ac35d21614", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:global-map:child-of:a88ca58757292f4c1c9b) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:global-map:child-of:a88ca58757292f4c1c9b\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:35:41+03:00\",\n    \"payload\": {\n      \"relationType\": \"CHILD_OF\",\n      \"sourceExternalKey\": \"domain:global:death-care-and-funeral-services\",\n      \"targetExternalKey\": \"category:global:personal-care-and-wellness\",\n      \"sourceEntityType\": \"DOMAIN\",\n      \"targetEntityType\": \"CATEGORY\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"Death Care & Funeral Services is commercially valid within Personal Care & Wellness; Core Domains may belong to multiple Categories.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:048f89a47eee166dad1fd6b6\",\n        \"sourceUrl\": \"https://www.census.gov/naics/?details=812&input=812&year=2022\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:35:41+03:00\",\n        \"claim\": \"2022 NAICS 812 — Personal and Laundry Services supports a commercially valid relationship between Core Domain Death Care & Funeral Services and Category Personal Care & Wellness.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "f5f969041765fc44f36f85135aaf6f3048b422b516b96e669bf932e33ba6c1e2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:belongs_to_category:7a5fd211b817f70948df) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:belongs_to_category:7a5fd211b817f70948df\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"BELONGS_TO_CATEGORY\",\n      \"sourceExternalKey\": \"family:axis:axis-m55-series\",\n      \"targetExternalKey\": \"category:security:ptz-camera\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"AXIS M55 Series is classified within PTZ Camera for this campaign.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:7445752a2722c9b21d1c\",\n        \"sourceUrl\": \"https://www.axis.com/products/ptz-cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"AXIS M55 Series is classified within PTZ Camera for this campaign.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "7fdaaa0d4a1aa945397a3f5876d77eea24c1ba2425d9a7bd1175956ec99d318d", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:hikvision:ds-81-recorder-family) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:hikvision:ds-81-recorder-family\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Hikvision\",\n      \"brand\": \"Hikvision\",\n      \"family\": \"DS-81 Recorder Family\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"lifecycle\": null,\n      \"name\": \"DS-81 Recorder Family\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:5b69d1e7a870eca0a874\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=3\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping DS-81 Recorder Family.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "fa359fade6a09f32f65b5b02d557cebb75e17b58eebb18d80081801e5b2f8c53", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=MANUFACTURER, externalKey=manufacturer:hanwha) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"MANUFACTURER\",\n    \"externalKey\": \"manufacturer:hanwha\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"name\": \"Hanwha Vision\",\n      \"countryOfOrigin\": \"South Korea\",\n      \"domain\": \"Security Systems\",\n      \"campaignRelevance\": \"Video surveillance, access control, or security-network infrastructure within Batch 001 scope.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:hanwha-qno6022r\",\n        \"sourceUrl\": \"https://supportportal.hanwhavision.com/global/products/QNO-6022R-en\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official Hanwha Vision source confirms relevant security/security-network product ecosystem.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b529efa8f6332bf7473bb21f002038218f9eb01a5e5cebb5837fc0487a657752", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:security-registry:parent_of:593fd4fb7bd0b88c9863) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:security-registry:parent_of:593fd4fb7bd0b88c9863\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"system:security:networked-access-control\",\n      \"targetExternalKey\": \"system:security:vehicle-gate-access-control\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"Networked Access Control System is the parent architecture for Vehicle Gate / Barrier Access Control System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:34647fb236ed69a563778492\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Networked Access Control System is the parent architecture for Vehicle Gate / Barrier Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "ad007a20fbae8254204badeda85bbff787e59d2b6292e886bf8d5eb89a7ebfca", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:security:commissioning) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:security:commissioning\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Security System Commissioning\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SERVICE\",\n      \"serviceClass\": \"COMMISSIONING\",\n      \"purpose\": \"Test and bring a newly deployed security system into operational service.\",\n      \"typicalActivities\": [\n        \"Perform commissioning tests\",\n        \"Validate subsystem operation\",\n        \"Address deployment issues\"\n      ],\n      \"typicalDeliverables\": [\n        \"Commissioned system / findings\"\n      ],\n      \"applicableSystemGroups\": [\n        \"ALL_SECURITY_SYSTEMS\"\n      ],\n      \"deliveryModes\": [\n        \"On-site\",\n        \"Remote\"\n      ],\n      \"recurring\": false,\n      \"prerequisites\": [],\n      \"vendorNeutralRegistryService\": true,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"service_class\",\n        \"name\": \"Service class\",\n        \"value\": \"COMMISSIONING\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:services:deployment\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recurring\",\n        \"name\": \"Recurring service\",\n        \"value\": false,\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"boolean\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:db47fcab0a4773bf80d8a67f\",\n        \"sourceUrl\": \"https://resources.genetec.com/i/1319812-genetec-professional-services\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Security System Commissioning.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "346905af6b8d1e5dd07f69ea6fcae51a7c1805134192099fe340c2af4191b3bd", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:hikvision:ds-90-recorder-family) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:hikvision:ds-90-recorder-family\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Hikvision\",\n      \"brand\": \"Hikvision\",\n      \"family\": \"DS-90 Recorder Family\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"lifecycle\": null,\n      \"name\": \"DS-90 Recorder Family\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:e7d8720bc34c9eef57f8\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=2\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping DS-90 Recorder Family.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "01c8e75488c4890828ae3777555f44c09745ead60ddaae2de0c4bef583a7495f", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:fleet-telematics) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:fleet-telematics\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Fleet Telematics\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Fleet Telematics is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on fleet telematics; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Automotive & Fleet\",\n        \"ICT & Technology\",\n        \"Logistics & Warehousing\",\n        \"Transportation & Mobility\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Driver monitoring\",\n        \"Fleet telemetry\",\n        \"Vehicle tracking\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"G\",\n      \"standardsOrClassificationEvidence\": [\n        \"UNGM Vehicles and Fleet Management\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:2f7d0ed93f8152c0586bb463\",\n        \"sourceUrl\": \"https://www.ungm.org/Shared/KnowledgeCenter/Pages/PC_VehiclesFleet\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNGM Vehicles and Fleet Management supports the technical/commercial scope represented by Core Domain Fleet Telematics.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:61c858ab3e25e57ec422039f\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Fleet Telematics.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "587b14db36d7f56f24df761dc1bc02492023e5a848f864c3dfd2c9d2a0c19776", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:parent_of:a4291d3c3f8c60bcc658) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:parent_of:a4291d3c3f8c60bcc658\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"family:i-pro:i-pro-s-series\",\n      \"targetExternalKey\": \"model:i-pro:i-pro-s-series:wv-s4176a\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"i-PRO S Series contains the verified model WV-S4176A.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:ca570b29f889990be307ed26\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=S-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"i-PRO S Series contains the verified model WV-S4176A.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a64aba5cbdbf146fae46993db81312f468b614afb9dcedfeeaf4fa906f7170cd", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:child_of:e59e5d88f83fa0392f36) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:child_of:e59e5d88f83fa0392f36\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"CHILD_OF\",\n      \"sourceExternalKey\": \"model:i-pro:i-pro-x-series:wv-x15501a-z3ln\",\n      \"targetExternalKey\": \"family:i-pro:i-pro-x-series\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"WV-X15501A-Z3LN is a child model of i-PRO X Series.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:6e2e373cc8c2166ba59501e8\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=X-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"WV-X15501A-Z3LN is a child model of i-PRO X Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "53de9c24696f4dc6696448b797e82d6cf8f6250735a0e809bcec925dbcde1043", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:security:requirements-assessment) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:security:requirements-assessment\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Security Requirements Assessment\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SERVICE\",\n      \"serviceClass\": \"CONSULTING\",\n      \"purpose\": \"Establish functional, operational and security requirements before system design.\",\n      \"typicalActivities\": [\n        \"Review business/security goals\",\n        \"Identify required functions and constraints\",\n        \"Document design inputs\"\n      ],\n      \"typicalDeliverables\": [\n        \"Requirements basis / assessment notes\"\n      ],\n      \"applicableSystemGroups\": [\n        \"ALL_SECURITY_SYSTEMS\"\n      ],\n      \"deliveryModes\": [\n        \"On-site\",\n        \"Remote\"\n      ],\n      \"recurring\": false,\n      \"prerequisites\": [],\n      \"vendorNeutralRegistryService\": true,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"service_class\",\n        \"name\": \"Service class\",\n        \"value\": \"CONSULTING\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:services:assessment-design\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recurring\",\n        \"name\": \"Recurring service\",\n        \"value\": false,\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"boolean\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:9851c0cef71e510286dccef3\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/solutions/professional-services\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Security Requirements Assessment.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "2efb084132e277984ada2e54da252bc1b0844ee2f832ddda5e2756e8ae10c99f", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:output-module) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:output-module\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"Output Module\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access I/O Module\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"Output Module\",\n      \"description\": \"Suprema current product line listed in the official hardware selector.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector2\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=2&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Output Module product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "e86d5933772bfddcf25a6b230058cf39b94347c7421886a296ffb3803ced7d84", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:hdcvi-surveillance) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:hdcvi-surveillance\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"HDCVI Surveillance System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"FORMAT_ARCHITECTURE\",\n      \"purpose\": \"Dahua-originated HD-over-coax surveillance architecture designed to extend and upgrade coax-based installations.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": \"HDCVI over coax\",\n      \"managementModel\": \"DVR/XVR-centric\",\n      \"recordingModel\": \"DVR/XVR-local\",\n      \"typicalComponents\": [\n        \"HDCVI cameras\",\n        \"Coaxial cabling\",\n        \"HDCVI-compatible DVR/XVR\",\n        \"Storage\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"HDCVI-compatible recorder input\"\n      ],\n      \"optionalComponents\": [\n        \"Power over Coax\",\n        \"IoT over coax\"\n      ],\n      \"protocols\": [\n        \"HDCVI\"\n      ],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"FORMAT_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:analog-coax-video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"signal_transport\",\n        \"name\": \"Signal / transport\",\n        \"value\": \"HDCVI over coax\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"management_model\",\n        \"name\": \"Management model\",\n        \"value\": \"DVR/XVR-centric\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recording_model\",\n        \"name\": \"Recording model\",\n        \"value\": \"DVR/XVR-local\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:116060d47772b2bce0a702e2\",\n        \"sourceUrl\": \"https://www.dahuasecurity.com/products/hdcvi-products/hdcvi-cameras\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by HDCVI Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "aabbb324e7ee8de9e646f477a9b5eade939b71a5e5fc8a1a555ccfb83c7a77be", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:port-handling-equipment) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:port-handling-equipment\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Port Handling Equipment\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Port Handling Equipment is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on port handling equipment; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Logistics & Warehousing\",\n        \"Marine & Maritime\",\n        \"Transportation & Mobility\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Container cranes\",\n        \"Reach stackers\",\n        \"Terminal handling\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"G\",\n      \"standardsOrClassificationEvidence\": [\n        \"UNGM Transportation and Storage Services\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:88d08f0eda906b982e3e4230\",\n        \"sourceUrl\": \"https://www.ungm.org/Shared/KnowledgeCenter/Pages/PC_TranStorMail\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNGM Transportation and Storage Services supports the technical/commercial scope represented by Core Domain Port Handling Equipment.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:ae94dce4e250fa3a9db51c16\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Port Handling Equipment.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "fccf19dc33a210800e97729031c9cc47b38289214472e85d3bf69ad92438afff", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=CATEGORY, externalKey=category:global:oil-gas-and-petrochemicals) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"CATEGORY\",\n    \"externalKey\": \"category:global:oil-gas-and-petrochemicals\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Oil, Gas & Petrochemicals\",\n      \"registryLayer\": \"CATEGORY\",\n      \"description\": \"Upstream, midstream, refining, gas/LNG and petrochemical commercial/engineering ecosystems.\",\n      \"commercialBoundary\": \"Separated from general energy because process equipment, hazardous-area systems and hydrocarbon logistics form a distinct market.\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"A\",\n      \"parentCategoryExternalKey\": null,\n      \"coreDomainCount\": 33,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [\n      \"Oil & Gas\",\n      \"Petroleum\",\n      \"Petrochemical\"\n    ],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:39c575ae8f93ddfa1e09385c\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the economic/procurement market scope used to define Oil, Gas & Petrochemicals.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:64d700dabdd332b5e0bd4fa7\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the economic/procurement market scope used to define Oil, Gas & Petrochemicals.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:9e3ec5df1f59f4a98fa290b2\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the economic/procurement market scope used to define Oil, Gas & Petrochemicals.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a79fd66492ad7dee8ea3d6a6b4e171869bb6e9935c012859828c0aee82197db2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:moxa:moxa-tn-g4500-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:moxa:moxa-tn-g4500-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Moxa\",\n      \"brand\": \"Moxa\",\n      \"family\": \"Moxa TN-G4500 Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Industrial PoE Switch\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"Moxa TN-G4500 Series\",\n      \"description\": \"Official Moxa industrial Ethernet series relevant to IP surveillance/security network transport.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"environment\",\n        \"name\": \"Target environment\",\n        \"value\": \"Railway / rugged transportation\",\n        \"unit\": null,\n        \"group\": \"Environmental\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"poe_standards\",\n        \"name\": \"PoE standards\",\n        \"value\": \"IEEE 802.3af/at\",\n        \"unit\": null,\n        \"group\": \"Ethernet/PoE\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"operating_temp_min\",\n        \"name\": \"Operating temperature minimum\",\n        \"value\": -40,\n        \"unit\": \"°C\",\n        \"group\": \"Environmental\",\n        \"dataType\": \"number\"\n      },\n      {\n        \"code\": \"operating_temp_max\",\n        \"name\": \"Operating temperature maximum\",\n        \"value\": 70,\n        \"unit\": \"°C\",\n        \"group\": \"Environmental\",\n        \"dataType\": \"number\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:moxa-tng4500\",\n        \"sourceUrl\": \"https://www.moxa.com/en/products/industrial-network-infrastructure/ethernet-switches/layer-2-managed-switches/tn-g4500-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Moxa TN-G4500 Series product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "3ced416a349bd7a74fe4ec5b3009d7dcf7739b42261041f87b801bb93120d507", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:tp-link:omada-access-pro-switches:sg3210x-m2) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:tp-link:omada-access-pro-switches:sg3210x-m2\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"TP-Link\",\n      \"brand\": \"Omada\",\n      \"family\": \"Omada Access Pro Switches\",\n      \"series\": \"Omada Access Pro Switches\",\n      \"model\": \"SG3210X-M2\",\n      \"modelNumber\": \"SG3210X-M2\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"SG3210X-M2\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Switch\",\n      \"subcategory\": null,\n      \"productRole\": \"Security network switch dependency\",\n      \"systemRole\": \"PoE Security Network Infrastructure\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"SG3210X-M2\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"SG3210X-M2\",\n        \"issuer\": \"TP-Link\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:876af5f774a0b71bde306d41\",\n        \"sourceUrl\": \"https://www.omadanetworks.com/us/business-networking/all-omada-switch/\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity SG3210X-M2.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "45799b848b9a042909648fe53994b0ed1b6429f691fac96c776119a299e33a73", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:moxa:eds-g205a-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:moxa:eds-g205a-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:51:44+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Moxa\",\n      \"brand\": \"Moxa\",\n      \"family\": \"EDS-G205A Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Industrial PoE Switch\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"EDS-G205A Series\",\n      \"description\": \"Official Moxa EDS-G205A Series product family.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"port_count\",\n        \"name\": \"Ethernet port count\",\n        \"value\": 5,\n        \"unit\": null,\n        \"group\": \"Interfaces\",\n        \"dataType\": \"number\"\n      },\n      {\n        \"code\": \"poe_ports\",\n        \"name\": \"PoE+ ports\",\n        \"value\": 4,\n        \"unit\": null,\n        \"group\": \"PoE\",\n        \"dataType\": \"number\"\n      },\n      {\n        \"code\": \"poe_standard\",\n        \"name\": \"PoE standard\",\n        \"value\": \"IEEE 802.3af/at\",\n        \"unit\": null,\n        \"group\": \"PoE\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:moxa-eds-g205a\",\n        \"sourceUrl\": \"https://www.moxa.com/en/products/industrial-network-infrastructure/ethernet-switches/poe-switches/eds-g205a-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:51:44+03:00\",\n        \"claim\": \"Official Moxa product page enumerates the available models and technical characteristics for EDS-G205A Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "226dc0fbb49aba8f9e09b8c5cac880183e326c819e3f24b9eb5fe35febbd613d", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:fiber-backbone-security-network) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:fiber-backbone-security-network\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Fiber-backbone Security Network\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"FIBER_NETWORK_ARCHITECTURE\",\n      \"purpose\": \"Security-network architecture using fiber uplinks/backbone between PoE access switches and control/recording locations.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"PoE access switches\",\n        \"SFP/fiber uplinks\",\n        \"Fiber plant\",\n        \"Core/aggregation equipment\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Optical transceiver/fiber compatibility\",\n        \"Link-budget design\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [\n        \"Ethernet over fiber\"\n      ],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"FIBER_NETWORK_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:security-networking\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:68064a89756a4f3b58fa935f\",\n        \"sourceUrl\": \"https://www.moxa.com/en/products/industrial-network-infrastructure/ethernet-switches/poe-switches/eds-p510a-series\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Fiber-backbone Security Network.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "ca265905ea3011c604ed7285d895e4be39f84f4afe5ae67fc9b118d28a530e88", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=MANUFACTURER, externalKey=manufacturer:axis) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"MANUFACTURER\",\n    \"externalKey\": \"manufacturer:axis\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"name\": \"Axis Communications\",\n      \"countryOfOrigin\": \"Sweden\",\n      \"domain\": \"Security Systems\",\n      \"campaignRelevance\": \"Video surveillance, access control, or security-network infrastructure within Batch 001 scope.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:axis-m30\",\n        \"sourceUrl\": \"https://www.axis.com/products/axis-m30-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official Axis Communications source confirms relevant security/security-network product ecosystem.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "24e0a29c81ce87e41e89a3e92f56f16807faea3dd04b85d456dd9698ec6fd22c", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:bispectral-thermal-visible-surveillance) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:bispectral-thermal-visible-surveillance\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Bispectral Thermal + Visible Surveillance System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"MULTISENSOR_ARCHITECTURE\",\n      \"purpose\": \"Integrated thermal-and-visible architecture combining heat-based detection with visual verification.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": \"Thermal + visible video\",\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Bispectral camera\",\n        \"VMS/NVR/cloud endpoint\",\n        \"Network\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Management support for selected bispectral device/functions\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"MULTISENSOR_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:specialized-video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"signal_transport\",\n        \"name\": \"Signal / transport\",\n        \"value\": \"Thermal + visible video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:c2c37b4cd66c1ca67f4d11f3\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/products/thermal-cameras\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Bispectral Thermal + Visible Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "344d7efe331d476e7da2457ae988c2ce510d72e928caff618d60dcb32f640a7a", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:door-module) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:door-module\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"Door Module\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access I/O Module\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"Door Module\",\n      \"description\": \"Suprema current product line listed in the official hardware selector.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector2\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=2&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Door Module product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "0aa0ab0f0565666b7c0efe7a6ace56a1375f6ce9d50fe2918257aed5126adf72", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:hikvision:ds-mp-recorder-family) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:hikvision:ds-mp-recorder-family\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Hikvision\",\n      \"brand\": \"Hikvision\",\n      \"family\": \"DS-MP Recorder Family\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"lifecycle\": null,\n      \"name\": \"DS-MP Recorder Family\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:7d6996b8b644389ad532\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=1\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping DS-MP Recorder Family.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "c04c8e0c5cd22a626829d51d7493a0ca3946a5f3bab36f5a0fa819f06c472cd2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:axis:axis-thermal-bispectral-cameras) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:axis:axis-thermal-bispectral-cameras\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Axis Communications\",\n      \"brand\": \"AXIS\",\n      \"family\": \"AXIS Thermal / Bispectral Cameras\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Thermal Camera\",\n      \"lifecycle\": null,\n      \"name\": \"AXIS Thermal / Bispectral Cameras\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:8d518f35af5dbfa40c15\",\n        \"sourceUrl\": \"https://www.axis.com/products/network-cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping AXIS Thermal / Bispectral Cameras.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "ff1b9c4e537fcb2844834a5cc241956f08e88588179f83843f51dcb2de69b5b5", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:edge-recording-ip-surveillance) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:edge-recording-ip-surveillance\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Edge-recording IP Surveillance System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"EDGE_RECORDING_ARCHITECTURE\",\n      \"purpose\": \"IP video architecture recording on the edge device or local edge storage rather than relying solely on a central recorder.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": \"Client/VMS\",\n      \"recordingModel\": \"Edge/device storage\",\n      \"typicalComponents\": [\n        \"Edge-recording cameras/devices\",\n        \"Local storage such as SD\",\n        \"Client/VMS\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Edge storage support and media\",\n        \"Retention/capacity design\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [\n        \"ONVIF Profile G where conformant\"\n      ],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"EDGE_RECORDING_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:ip-video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"management_model\",\n        \"name\": \"Management model\",\n        \"value\": \"Client/VMS\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recording_model\",\n        \"name\": \"Recording model\",\n        \"value\": \"Edge/device storage\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:c54f89470b7a7d60ff416f65\",\n        \"sourceUrl\": \"https://www.onvif.org/profiles/profile-g/\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Edge-recording IP Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:security-registry:fc95a6a122cb27e69ebcf5af\",\n        \"sourceUrl\": \"https://www.axis.com/products/axis-camera-station-edge\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Edge-recording IP Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "6462d145e3203cc278b32cf86411e9633a55b38291a7b8b73d5643b0f0eaee64", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:axis:axis-p47-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:axis:axis-p47-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Axis Communications\",\n      \"brand\": \"AXIS\",\n      \"family\": \"AXIS P47 Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Panoramic Camera\",\n      \"lifecycle\": null,\n      \"name\": \"AXIS P47 Series\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:e56e212a4f877e84d9be\",\n        \"sourceUrl\": \"https://www.axis.com/products/network-cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping AXIS P47 Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "f79093858d95484630c986501f3f7536766fa68f636a0b1df640673372d18fc7", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:child_of:a3c6ba2365d9eb9d682b) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:child_of:a3c6ba2365d9eb9d682b\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"CHILD_OF\",\n      \"sourceExternalKey\": \"model:i-pro:i-pro-x-series:wv-x25700a-v2ln\",\n      \"targetExternalKey\": \"family:i-pro:i-pro-x-series\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"WV-X25700A-V2LN is a child model of i-PRO X Series.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:4da51ac94e4c77e0bcdde364\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=X-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"WV-X25700A-V2LN is a child model of i-PRO X Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b12a115a64cb1462d3d58c8eb24228ced20fa1dab238ab33e0931e0d98ea4f06", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:belongs_to_category:87008cccd9f17d93) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:belongs_to_category:87008cccd9f17d93\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BELONGS_TO_CATEGORY\",\n      \"sourceExternalKey\": \"model:suprema:door-interface:di-24\",\n      \"targetExternalKey\": \"category:security:access-io-module\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector3\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=3&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BELONGS_TO_CATEGORY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "bc6b27535c32ccad2181688c5a9dbc2b73ae484332dc4165ab03b4d383eccdd5", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=CATEGORY, externalKey=category:global:agriculture-and-agribusiness) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"CATEGORY\",\n    \"externalKey\": \"category:global:agriculture-and-agribusiness\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Agriculture & Agribusiness\",\n      \"registryLayer\": \"CATEGORY\",\n      \"description\": \"Production, handling and commercial systems for crops, livestock, forestry and agricultural operations.\",\n      \"commercialBoundary\": \"Excludes downstream industrial food processing when the main activity is manufacturing rather than primary agriculture.\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"A\",\n      \"parentCategoryExternalKey\": null,\n      \"coreDomainCount\": 13,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [\n      \"Agriculture\",\n      \"Agribusiness\"\n    ],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:5d9274fae97bea2eec53af8b\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the economic/procurement market scope used to define Agriculture & Agribusiness.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:dc501fd3ca87942e3b2858a3\",\n        \"sourceUrl\": \"https://www.census.gov/naics/\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"2022 North American Industry Classification System supports the economic/procurement market scope used to define Agriculture & Agribusiness.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:c6182860157e4cfd1d146b40\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the economic/procurement market scope used to define Agriculture & Agribusiness.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "baddf953b98db8c6817bebb6a009f7636bb42afd20e8b32e3bdfa50455be2195", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:high-assurance-encrypted-access-control) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:high-assurance-encrypted-access-control\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"High-Assurance Encrypted Access Control System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"HIGH_ASSURANCE_ARCHITECTURE\",\n      \"purpose\": \"End-to-end protected access architecture using encrypted credentials, readers/interfaces, secure I/O/gateways and access software.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Access software\",\n        \"Secure gateway/controller\",\n        \"Secure I/O\",\n        \"Transparent/secure reader\",\n        \"Smart credential\",\n        \"Secure Access Module where required\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Validated component chain and cryptographic configuration\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [\n        \"TLS\",\n        \"OSDP Secure Channel where applicable\"\n      ],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"HIGH_ASSURANCE_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:access-control\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:3d7bf0f932fb0a413ed3af82\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis/high-assurance-access-control\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by High-Assurance Encrypted Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "7ea389c99aef08f209c3bb579658d6cb1124eeacbc042c491ab1527562461fb6", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:electronic-article-surveillance-and-loss-prevention) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:electronic-article-surveillance-and-loss-prevention\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Electronic Article Surveillance & Loss Prevention\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Electronic Article Surveillance & Loss Prevention is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on electronic article surveillance & loss prevention; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Retail & Commerce\",\n        \"Security & Public Safety\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Asset protection\",\n        \"EAS\",\n        \"Retail loss prevention\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"D\",\n      \"standardsOrClassificationEvidence\": [\n        \"UNGM Public Order, Security and Safety Services\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:fc53678dc92bd9e5ee36ac4e\",\n        \"sourceUrl\": \"https://www.ungm.org/Shared/KnowledgeCenter/Pages/PC_Security\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNGM Public Order, Security and Safety Services supports the technical/commercial scope represented by Core Domain Electronic Article Surveillance & Loss Prevention.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:6e18f174a16a06f9636cff86\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Electronic Article Surveillance & Loss Prevention.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a745f758cec35f90d11adb1190baa29031364162c3d538efa96e6e34cc1db255", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:suprema-discontinued-access-products) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:suprema-discontinued-access-products\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"Suprema Discontinued Access Products\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access Control\",\n      \"lifecycle\": \"LEGACY\",\n      \"name\": \"Suprema Discontinued Access Products\",\n      \"description\": \"Official Suprema discontinued-products area lists these historical access-control products.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-eol\",\n        \"sourceUrl\": \"https://www.supremainc.com/en/hardware/eol_biostation.asp\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Suprema Discontinued Access Products product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "cb5e14effc8cf912a5de8dce0599968fd1495afef456b9052acc494b4bba8cd1", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=CATEGORY, externalKey=category:global:textiles-and-apparel) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"CATEGORY\",\n    \"externalKey\": \"category:global:textiles-and-apparel\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Textiles & Apparel\",\n      \"registryLayer\": \"CATEGORY\",\n      \"description\": \"Textile, garment, workwear and related manufacturing/product markets.\",\n      \"commercialBoundary\": \"Technical textiles can overlap Construction, Healthcare and Industrial.\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"parentCategoryExternalKey\": null,\n      \"coreDomainCount\": 3,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [\n      \"Textiles\",\n      \"Apparel\",\n      \"Clothing\"\n    ],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:8045125bab4b8b79c811ecac\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the economic/procurement market scope used to define Textiles & Apparel.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:8549b94b6a56b24bc694e92b\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the economic/procurement market scope used to define Textiles & Apparel.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:3822810311cf93d98a937cb6\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the economic/procurement market scope used to define Textiles & Apparel.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a00bf284307ea606334c052f669f518f80e8991f1d2b5c3c9ef0f854bad9f810", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:dahua:dahua-discontinued-network-cameras:ipc-hfw21249t-s-il) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:dahua:dahua-discontinued-network-cameras:ipc-hfw21249t-s-il\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Dahua Technology\",\n      \"brand\": \"Dahua\",\n      \"family\": \"Dahua Discontinued Network Cameras\",\n      \"series\": \"Dahua Discontinued Network Cameras\",\n      \"model\": \"IPC-HFW21249T-S-IL\",\n      \"modelNumber\": \"IPC-HFW21249T-S-IL\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"IPC-HFW21249T-S-IL\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"Historical network surveillance camera\",\n      \"systemRole\": \"IP Video Surveillance System\",\n      \"lifecycle\": \"LEGACY\",\n      \"name\": \"IPC-HFW21249T-S-IL\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"IPC-HFW21249T-S-IL\",\n        \"issuer\": \"Dahua Technology\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:73a02b3c3478232ec8fc460b\",\n        \"sourceUrl\": \"https://www.dahuasecurity.com/sa/Products/All-Products/Discontinued-Products/Network-Cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity IPC-HFW21249T-S-IL.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "3ecaa5bf2a483e759828185538d8c852161e8dd48d73a545d2accfcf4899d9cf", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:tp-link:vigi-surveillance-kits) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:tp-link:vigi-surveillance-kits\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"TP-Link\",\n      \"brand\": \"VIGI\",\n      \"family\": \"VIGI Surveillance Kits\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Surveillance\",\n      \"lifecycle\": null,\n      \"name\": \"VIGI Surveillance Kits\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:1ab9db79aab2ef3751fe\",\n        \"sourceUrl\": \"https://www.vigi.com/us/vigi-app/product-list/\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping VIGI Surveillance Kits.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "8eef568bf2f88d75e0639022e57d86c70d085bc6ce27eedd0de0a67cf7898a8e", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:cranes-and-lifting-equipment) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:cranes-and-lifting-equipment\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Cranes & Lifting Equipment\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Cranes & Lifting Equipment is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on cranes & lifting equipment; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Construction\",\n        \"Industrial & Manufacturing\",\n        \"Logistics & Warehousing\",\n        \"Marine & Maritime\",\n        \"Mining & Minerals\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Hoists\",\n        \"Mobile lifting\",\n        \"Overhead cranes\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:513b170e8621f73d39633aa3\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Cranes & Lifting Equipment.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:c40d43bedb9adf3b5e27d8d0\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Cranes & Lifting Equipment.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "922a508d37f1e36c8a294d20a5ba55705fe7408d500e3b9e0696bf0e72959eb1", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:corestation) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:corestation\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"CoreStation\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access Controller\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"CoreStation\",\n      \"description\": \"Suprema current product line listed in the official hardware selector.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector2\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=2&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the CoreStation product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "5b77b0a4e84e3817a046d3319f112e862c4045d546c592b6b15901ae9aa1371e", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:i-pro:i-pro-s-series:wv-s66300-z4ln) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:i-pro:i-pro-s-series:wv-s66300-z4ln\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"i-PRO\",\n      \"brand\": \"i-PRO\",\n      \"family\": \"i-PRO S Series\",\n      \"series\": \"i-PRO S Series\",\n      \"model\": \"WV-S66300-Z4LN\",\n      \"modelNumber\": \"WV-S66300-Z4LN\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"WV-S66300-Z4LN\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"Network camera\",\n      \"systemRole\": \"IP Video Surveillance System\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"WV-S66300-Z4LN\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"WV-S66300-Z4LN\",\n        \"issuer\": \"i-PRO\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:615e2583148ac86ef2089a60\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=S-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity WV-S66300-Z4LN.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "8babd32b23f68f892a11a994354bb0b9f31faf8311baed14f958d57f100ef48b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:b3a3d33af19ba0fb) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:b3a3d33af19ba0fb\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:ubiquiti:unifi-access-current-readers-and-intercoms:g3-intercom\",\n      \"targetExternalKey\": \"brand:unifi\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:ubi-access-readers\",\n        \"sourceUrl\": \"https://www.ui.com/us/en/door-access/readers\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BRANDED_BY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "390eb3c17592abd6c65041718f0155a44e51d7718523cd0d8b5996330374d33b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:i-pro:i-pro-s-series:wv-s32402-f2l) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:i-pro:i-pro-s-series:wv-s32402-f2l\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"i-PRO\",\n      \"brand\": \"i-PRO\",\n      \"family\": \"i-PRO S Series\",\n      \"series\": \"i-PRO S Series\",\n      \"model\": \"WV-S32402-F2L\",\n      \"modelNumber\": \"WV-S32402-F2L\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"WV-S32402-F2L\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"Network camera\",\n      \"systemRole\": \"IP Video Surveillance System\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"WV-S32402-F2L\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"WV-S32402-F2L\",\n        \"issuer\": \"i-PRO\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:63c47e91eb720f4dc95c8f6c\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=S-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity WV-S32402-F2L.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "8283417e7793a6f14a58997aae2a244da0903f89680dfe09d081530e6939a62b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=BRAND, externalKey=brand:hid-mercury) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"BRAND\",\n    \"externalKey\": \"brand:hid-mercury\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"name\": \"HID Mercury\",\n      \"manufacturerExternalKey\": \"manufacturer:hid\",\n      \"domain\": \"Security Systems\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:hid-certs\",\n        \"sourceUrl\": \"https://www.hidglobal.com/certifications\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official manufacturer source identifies the HID Mercury product ecosystem.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}]''')
print('qualification prompts:', len(PROMPTS))
assert all(set(p.keys()) == {'item_id', 'messages'} for p in PROMPTS), 'gold leaked into the prompt payload'
print('payload contains ONLY item_id + messages (no gold)')

WORKING = pathlib.Path('/kaggle/working')
ADAPTER = None
for cand in sorted(pathlib.Path('/kaggle/input').rglob('adapter_model.safetensors')):
    if 'checkpoint' not in str(cand):
        ADAPTER = cand.parent
        break
assert ADAPTER is not None, 'adapter not found under /kaggle/input'
print('adapter dir:', ADAPTER)
print('adapter bytes:', (ADAPTER / 'adapter_model.safetensors').stat().st_size)

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER), max_seq_length=3072, load_in_4bit=True, full_finetuning=False,
)
FastLanguageModel.for_inference(model)

def render(messages):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        reasoning_effort='medium', strftime_now=lambda _f: '2026-09-15',
    )

def final_channel(text):
    """Deterministic final-channel extraction; analysis is NEVER returned."""
    import re
    last = None
    for m in re.finditer(r'<\|channel\|>([A-Za-z_][A-Za-z0-9_]*)\s*<\|message\|>', text):
        start = m.end(); end = len(text)
        for tok in ('<|return|>', '<|end|>', '<|call|>', '<|start|>'):
            i = text.find(tok, start)
            if i != -1: end = min(end, i)
        if m.group(1) == 'final': last = text[start:end]
    return last

preds = []
for i, item in enumerate(PROMPTS):
    ids = tokenizer(render(item['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=3072, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)
    ans = final_channel(raw)
    preds.append({'item_id': item['item_id'], 'raw': raw, 'prediction': ans,
                  'ok': ans is not None, 'analysis_present': '<|channel|>analysis' in raw})
    if i % 10 == 0: print('generated', i + 1, '/', len(PROMPTS), '| ok:', ans is not None)

(WORKING / 'predictions.jsonl').write_text(
    '\n'.join(json.dumps(p, ensure_ascii=False) for p in preds) + '\n', encoding='utf-8')
summary = {'items': len(preds), 'ok': sum(1 for p in preds if p['ok']),
           'no_final_channel': sum(1 for p in preds if not p['ok']),
           'analysis_present_count': sum(1 for p in preds if p['analysis_present']),
           'adapter': str(ADAPTER)}
(WORKING / 'inference-summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2)); print('COMPLETE')
